In [ ]:
# Parameters
function_type = "Balanced-Identity"


In [ ]:
%matplotlib inline
import matplotlib
matplotlib.use('agg')
import warnings
warnings.filterwarnings('ignore')


In [ ]:

from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer
from qiskit.visualization import circuit_drawer, plot_histogram, plot_bloch_multivector
from qiskit.quantum_info import Statevector
import matplotlib.pyplot as plt
import numpy as np

function_type = str(function_type)

qc = QuantumCircuit(2, 1)
# Prepare |01⟩ then apply H to both
qc.x(1)
qc.barrier()
qc.h(0)
qc.h(1)
qc.barrier()

# Oracle Uf
if function_type == "Constant-0":
    pass  # Identity
elif function_type == "Constant-1":
    qc.x(1)
elif function_type == "Balanced-Identity":
    qc.cx(0, 1)
elif function_type == "Balanced-Negation":
    qc.cx(0, 1)
    qc.x(1)

qc.barrier()
qc.h(0)
qc.measure(0, 0)

print(f"Deutsch Algorithm — Function: {function_type}")

fig = circuit_drawer(qc, output='mpl')
display(fig)
plt.close(fig)

simulator = Aer.get_backend('aer_simulator')
compiled = transpile(qc, simulator)
job = simulator.run(compiled, shots=1000)
result = job.result()
counts = result.get_counts()

is_balanced = '1' in counts and counts.get('1', 0) > counts.get('0', 0)
print(f"Result: Function is {'BALANCED' if is_balanced else 'CONSTANT'}")
print(f"Counts: {counts}")

fig2 = plot_histogram(counts)
display(fig2)
plt.close(fig2)

# Bloch sphere for qubit 0
qc_sv = QuantumCircuit(2)
qc_sv.x(1)
qc_sv.h(0)
qc_sv.h(1)
if function_type == "Constant-0":
    pass
elif function_type == "Constant-1":
    qc_sv.x(1)
elif function_type == "Balanced-Identity":
    qc_sv.cx(0, 1)
elif function_type == "Balanced-Negation":
    qc_sv.cx(0, 1)
    qc_sv.x(1)
qc_sv.h(0)
sv = Statevector.from_instruction(qc_sv)
from qiskit.quantum_info import partial_trace
rho = partial_trace(sv, [1])
a = np.sqrt(np.real(rho.data[0, 0]))
b_complex = rho.data[1, 0] / a if a > 1e-6 else 0
theta = 2 * np.arccos(np.clip(a, 0, 1))
phi = np.angle(b_complex) % (2 * np.pi)

try:
    fig3 = plot_bloch_multivector(sv)
    display(fig3)
    plt.close(fig3)
except Exception:
    pass
print(f"BLOCH_THETA={theta:.6f}")
print(f"BLOCH_PHI={phi:.6f}")
